# <br><center> Exploring the paired T-test using cross validation

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.figsize": (14,8),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.2
})

import scipy.stats as stats
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.datasets import load_diabetes

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

from dotenv import load_dotenv
load_dotenv()

import gc; gc.enable()
from IPython.display import clear_output, display
from tqdm.notebook import tqdm

print("setup done!")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# load data
ds = load_diabetes()
desc = ds["DESCR"]

X = ds["data"]
y = ds["target"]

del ds; gc.collect()
print(desc)

In [ ]:
# cross validation
model1 = XGBRegressor(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=3,
    early_stopping_rounds=30,
    random_state=67
)

model2 = CatBoostRegressor(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=3,
    early_stopping_rounds=30,
    random_state=67,
)

n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=67)

xs = np.zeros(n_splits)
ys = np.zeros(n_splits)
for fold, (train_idx, test_idx) in enumerate(tqdm(kf.split(X), desc="Model Evaluation", leave=False)):
    x_train, y_train = X[train_idx], y[train_idx]
    x_test, y_test = X[test_idx], y[test_idx]

    model1.fit(x_train, y_train, eval_set=[(x_train, y_train), (x_test, y_test)], verbose=100)
    model2.fit(x_train, y_train, eval_set=[(x_train, y_train), (x_test, y_test)], verbose=100)

    y_pred_1 = model1.predict(x_test)
    y_pred_2 = model2.predict(x_test)

    xs[fold] = root_mean_squared_error(y_test, y_pred_1)
    ys[fold] = root_mean_squared_error(y_test, y_pred_2)


In [ ]:
plt.plot(xs, color="green", label="Model 1 RMSE")
plt.plot(ys, color="maroon", label="Model 2 RMSE")
plt.title("Cross validation error for xgboost (1) and catboost (2)")
plt.xlabel("Fold")
plt.ylabel("Root Mean Squared Error")
plt.legend()
plt.show()

In [ ]:
# paired t-test
results = stats.ttest_1samp(xs - ys, popmean=0)
print(f"T-test results: t={results.statistic:.4f}, p={results.pvalue:.8f}, df={int(results.df)}")

### Conclusion 1
Looking at the results of the t-test, we see that as our $p$-value was greater than 0.05, the experiment is inconclusive as of now, <br> and we need more data to effectively support that one model is better than the other.

In [ ]:
# cross validation
model1 = XGBRegressor(
    early_stopping_rounds=30,
    random_state=67
)

model2 = LGBMRegressor(
    early_stopping_rounds=30,
    random_state=67,
    verbose=-100
)

n_splits = 25
kf = KFold(n_splits=n_splits, shuffle=True, random_state=67)

xs = np.zeros(n_splits)
ys = np.zeros(n_splits)
for fold, (train_idx, test_idx) in enumerate(tqdm(kf.split(X), desc="Model Evaluation", leave=False)):
    x_train, y_train = X[train_idx], y[train_idx]
    x_test, y_test = X[test_idx], y[test_idx]

    model1.fit(x_train, y_train, eval_set=[(x_train, y_train), (x_test, y_test)], verbose=100)
    model2.fit(x_train, y_train, eval_X=(x_train, x_test), eval_y=(y_train, y_test))

    y_pred_1 = model1.predict(x_test)
    y_pred_2 = model2.predict(x_test)

    xs[fold] = root_mean_squared_error(y_test, y_pred_1)
    ys[fold] = root_mean_squared_error(y_test, y_pred_2)

In [ ]:
plt.plot(xs, color="green", label="Model 1 RMSE")
plt.plot(ys, color="maroon", label="Model 2 RMSE")
plt.title("Cross validation error for xgboost (1) and catboost (2)")
plt.xlabel("Fold")
plt.ylabel("Root Mean Squared Error")
plt.legend()
plt.show()

In [ ]:
# paired t-test
results = stats.ttest_1samp(xs - ys, popmean=0)
print(f"T-test results: t={results.statistic:.4f}, p={results.pvalue:.8f}, df={int(results.df)}")

### Conclusion 2
Looking now at the new data, we see that in our new experiment, the lightgbm regressor outperforms the XGBRegressor as suggested by our t-test.

The $p$-value suggests extremely high levels of statistical significance, which is strong evidence that the gradient booster performed best on this dataset.